In [21]:
from utils.helpers import load_dataset
from utils.randomizer import (
    validate_dataset,
    shuffle_options_in_item,
    randomize_dataset,
    verify_randomized_dataset,
    export_jsonl,
)

import json

In [4]:
INPUT_PATH = r"data\information-structure\eval\inf_structure_eval_720.json"
OUTPUT_JSON_PATH = r"data\information-structure\eval\randomized_inf_structure_eval_720.json"
OUTPUT_JSONL_PATH = r"data\information-structure\eval\final_randomized_inf_structure_eval_720.json"

MASTER_SEED = 42
CORRECT_LABEL_FIELD = "gold_label"   # or "correct_option" if your dataset still uses that

In [5]:
items = load_dataset(INPUT_PATH)

print(f"loaded {len(items)} items.")
print(f"top-level type: {type(items).__name__}")
print(f"first item id: {items[0]['id']}")

loaded 720 items.
top-level type: list
first item id: information-structure-17-focus_object


In [6]:
# validating if the dataset structure correspods to the needed standardized structure
errors = validate_dataset(items, correct_label_field=CORRECT_LABEL_FIELD)

print(f"Validation errors found: {len(errors)}")

if errors:
    print("\nFirst 20 errors:")
    for err in errors[:20]:
        print("-", err)
else:
    print("Dataset validation passed.")

Validation errors found: 213

First 20 errors:
- Duplicate item id found: 'information-structure-17-focus_object'.
- Duplicate item id found: 'information-structure-17-focus_subject'.
- Duplicate item id found: 'information-structure-17-focus_verb'.
- Duplicate item id found: 'information-structure-18-focus_object'.
- Duplicate item id found: 'information-structure-18-focus_subject'.
- Duplicate item id found: 'information-structure-18-focus_verb'.
- Duplicate item id found: 'information-structure-19-focus_object'.
- Duplicate item id found: 'information-structure-19-focus_subject'.
- Duplicate item id found: 'information-structure-19-focus_verb'.
- Duplicate item id found: 'information-structure-20-focus_object'.
- Duplicate item id found: 'information-structure-20-focus_subject'.
- Duplicate item id found: 'information-structure-20-focus_verb'.
- Duplicate item id found: 'information-structure-21-focus_object'.
- Duplicate item id found: 'information-structure-21-focus_subject'.
- Du

## testing answer options shuffle on a single item

In [12]:
sample_item = items[0]

print("ID:", sample_item["id"])
print("gold label:", sample_item[CORRECT_LABEL_FIELD])
print("\noptions before randomization:")

for opt in sample_item["options"]:
    marker = " <-- GOLD LABEL" if opt["label"] == sample_item[CORRECT_LABEL_FIELD] else ""
    print(f"{opt['label']}. [{opt['type']}] {opt['text']}{marker}")

ID: information-structure-17-focus_object
gold label: A

options before randomization:
A. [object_last] Kuchař připálil omáčku. <-- GOLD LABEL
B. [subject_last] Omáčku připálil kuchař.
C. [verb_last] Kuchař omáčku připálil.
D. [distractor] Kuchař otevřel troubu.


In [13]:
shuffled_sample = shuffle_options_in_item(
    sample_item,
    master_seed=MASTER_SEED,
    correct_label_field=CORRECT_LABEL_FIELD,
)

print("ID:", shuffled_sample["id"])
print("gold label after shuffle:", shuffled_sample[CORRECT_LABEL_FIELD])
print("\noptions after randomization:")

for opt in shuffled_sample["options"]:
    marker = " <-- GOLD" if opt["label"] == shuffled_sample[CORRECT_LABEL_FIELD] else ""
    print(f"{opt['label']}. [{opt['type']}] {opt['text']}{marker}")

ID: information-structure-17-focus_object
gold label after shuffle: C

options after randomization:
A. [subject_last] Omáčku připálil kuchař.
B. [distractor] Kuchař otevřel troubu.
C. [object_last] Kuchař připálil omáčku. <-- GOLD
D. [verb_last] Kuchař omáčku připálil.


## randomizing

In [14]:
# now we can randomize the whole dataset (shuffle items + opions in each item) and see the result
randomized_items = randomize_dataset(
    items,
    master_seed=MASTER_SEED,
    correct_label_field=CORRECT_LABEL_FIELD,
)

print(f"original dataset size: {len(items)}")
print(f"randomized dataset size: {len(randomized_items)}")

original dataset size: 720
randomized dataset size: 720


## observing randomization results + verification

In [16]:
print("first 10 original item IDs:")
for item in items[:10]:
    print(item["id"])

print("\nfirst 10 randomized item IDs:")
for item in randomized_items[:10]:
    print(item["id"])

first 10 original item IDs:
information-structure-17-focus_object
information-structure-17-focus_subject
information-structure-17-focus_verb
information-structure-18-focus_object
information-structure-18-focus_subject
information-structure-18-focus_verb
information-structure-19-focus_object
information-structure-19-focus_subject
information-structure-19-focus_verb
information-structure-20-focus_object

first 10 randomized item IDs:
information-structure-122-focus_verb
information-structure-124-focus_verb
information-structure-55-focus_subject
information-structure-118-focus_object
information-structure-38-focus_object
information-structure-80-focus_subject
information-structure-154-focus_subject
information-structure-138-focus_subject
information-structure-70-focus_object
information-structure-29-focus_object


In [17]:
problems = verify_randomized_dataset(
    items,
    randomized_items,
    correct_label_field=CORRECT_LABEL_FIELD,
)

print(f"verification problems found: {len(problems)}")

if problems:
    print("\nfirst 20 problems:")
    for problem in problems[:20]:
        print("-", problem)
else:
    print("post-randomization verification passed.")

verification problems found: 228

first 20 problems:
- Item 'information-structure-17-focus_object': option contents changed after randomization.
- Item 'information-structure-17-focus_object': gold option meaning changed after randomization.
- Item 'information-structure-17-focus_subject': option contents changed after randomization.
- Item 'information-structure-17-focus_subject': gold option meaning changed after randomization.
- Item 'information-structure-18-focus_object': option contents changed after randomization.
- Item 'information-structure-18-focus_object': gold option meaning changed after randomization.
- Item 'information-structure-18-focus_subject': option contents changed after randomization.
- Item 'information-structure-18-focus_subject': gold option meaning changed after randomization.
- Item 'information-structure-19-focus_object': option contents changed after randomization.
- Item 'information-structure-19-focus_object': gold option meaning changed after randomiz

In [19]:
# quick check to verify reproducibility: randomizing with the same seed returns the same result
randomized_items_again = randomize_dataset(
    items,
    master_seed=MASTER_SEED,
    correct_label_field=CORRECT_LABEL_FIELD,
)

print("verification: identical with same seed:", randomized_items == randomized_items_again)

verification: identical with same seed: True


In [ ]:
sample_id = items[0]["id"]

orig_by_id = {item["id"]: item for item in items}
rand_by_id = {item["id"]: item for item in randomized_items}

orig_item = orig_by_id[sample_id]
rand_item = rand_by_id[sample_id]

print("ORIGINAL")
print("ID:", orig_item["id"])
print("Gold label:", orig_item[CORRECT_LABEL_FIELD])
for opt in orig_item["options"]:
    marker = " <-- GOLD" if opt["label"] == orig_item[CORRECT_LABEL_FIELD] else ""
    print(f"{opt['label']}. [{opt['type']}] {opt['text']}{marker}")

print("\nRANDOMIZED")
print("ID:", rand_item["id"])
print("Gold label:", rand_item[CORRECT_LABEL_FIELD])
for opt in rand_item["options"]:
    marker = " <-- GOLD" if opt["label"] == rand_item[CORRECT_LABEL_FIELD] else ""
    print(f"{opt['label']}. [{opt['type']}] {opt['text']}{marker}")

## save results to files

In [ ]:
with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(randomized_items, f, ensure_ascii=False, indent=2)

print(f"Saved randomized JSON to: {OUTPUT_JSON_PATH}")
print(f"Master seed: {MASTER_SEED}")

In [ ]:
export_jsonl(randomized_items, OUTPUT_JSONL_PATH)
print(f"Saved randomized JSONL to: {OUTPUT_JSONL_PATH}")